<a href="https://colab.research.google.com/github/rjortega-dev/DS-LLM-Lab8/blob/main/DS_LLM_SummativeLab8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Summative Lab

MidCity General Hospital is facing challenges with bed management and resource allocation. The hospital administration has commissioned your data science team to develop a predictive model to estimate patient length of stay (LOS) at admission, which will help the hospital:

* Optimize staffing levels across departments.
* Improve bed management and patient flow.
* Reduce costs while maintaining the quality of care.
* Identify factors that contribute to extended hospital stays.

You have been provided with historical patient data from the past year, including both structured patient information and unstructured clinical notes. Your task is to develop two different approaches to predict the length of stay and compare/combine their effectiveness.



---



## Part 1: Using structured patient data with traditional ML models.



### Task 1: Setup and Data Exploration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('hospital_patient_data.csv')

print(df.shape)
display(df.head())
df.info()
print(df.isna().sum())
display(df.describe(include='all').T)

(10000, 31)


,patient_id,age,gender,race_ethnicity,marital_status,insurance_type,employment_status,education_level,income_bracket,primary_diagnosis,...,temperature,respiratory_rate,hospital_department,attending_physician_id,surgery_performed,icu_stay,date_of_admission,date_of_discharge,readmission_30day,patient_notes
0,P000001,65,Female,Caucasian,Single,Medicare,Unknown,Bachelor's Degree,"$75,000-$100,000",Heart Failure,...,36.6,12,Orthopedics,DRO6,False,True,2024-09-19,2024-09-30,False,Patient is a 65-year-old female presenting wit...
1,P000002,91,Male,African American,Widowed,Private,Retired,High School,"$25,000-$50,000",Diabetes with Complications,...,37.7,16,Gastroenterology,DRG4,False,False,2025-04-14,2025-04-21,False,Patient is a 91-year-old male presenting with ...
2,P000003,49,Male,Hispanic/Latino,Widowed,Private,Unemployed,Bachelor's Degree,"$25,000-$50,000",Fracture of Femur,...,36.8,21,Nephrology,DRN7,False,False,2024-12-21,2024-12-28,False,"male patient, 49, presents with swelling, leg ..."
3,P000004,28,Female,Caucasian,Married,Medicare,Retired,Some College,"$25,000-$50,000",Kidney Infection,...,38.0,17,Neurology,DRN7,True,False,2024-07-17,2024-07-23,False,Patient is a 28-year-old female presenting wit...
4,P000005,74,Male,Caucasian,Separated,Other,Employed,High School,"$75,000-$100,000",Kidney Infection,...,36.7,17,Orthopedics,DRO7,True,False,2025-03-25,2025-04-02,False,"male patient, 74, presents with fever, dysuria..."


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   patient_id              10000 non-null  object 
 1   age                     10000 non-null  int64  
 2   gender                  10000 non-null  object 
 3   race_ethnicity          10000 non-null  object 
 4   marital_status          10000 non-null  object 
 5   insurance_type          10000 non-null  object 
 6   employment_status       10000 non-null  object 
 7   education_level         10000 non-null  object 
 8   income_bracket          10000 non-null  object 
 9   primary_diagnosis       10000 non-null  object 
 10  diagnosis_code          10000 non-null  object 
 11  comorbidities           8033 non-null   object 
 12  emergency_admission     10000 non-null  bool   
 13  previous_admissions     10000 non-null  int64  
 14  medications_count       10000 non-null 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
patient_id,10000,10000,P000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,10000.0,NaN,NaN,NaN,59.0886,20.075105,18.0,45.0,60.0,75.0,94.0
gender,10000,3,Female,5062,NaN,NaN,NaN,NaN,NaN,NaN,NaN
race_ethnicity,10000,6,Caucasian,6025,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital_status,10000,6,Married,3950,NaN,NaN,NaN,NaN,NaN,NaN,NaN
insurance_type,10000,5,Private,3581,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employment_status,10000,5,Employed,4512,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education_level,10000,6,Some College,2513,NaN,NaN,NaN,NaN,NaN,NaN,NaN
income_bracket,10000,6,"$50,000-$75,000",2489,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_diagnosis,10000,18,Heart Failure,1045,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Step 2: Build the target and sanity-check it

In [2]:
# Parse dates and create target
df['date_of_admission'] = pd.to_datetime(df['date_of_admission'])
df['date_of_discharge'] = pd.to_datetime(df['date_of_discharge'])
df['los'] = (df['date_of_discharge'] - df['date_of_admission']).dt.days

print(df['los'].describe())
print((df['los'] <= 0).sum(), "rows with LOS <= 0")
print(df[['date_of_admission', 'date_of_discharge']].agg(['min', 'max']))
print(df['los'].value_counts().sort_index())

# Group sizes (fairness groups)
for c in ['gender', 'race_ethnicity', 'insurance_type', 'income_bracket', 'alcohol_use']:
    print(df[c].value_counts(dropna=False), '\n')

# Does the "during/after stay" info track LOS?
for c in ['icu_stay', 'surgery_performed', 'readmission_30day', 'emergency_admission']:
    print(df.groupby(c)['los'].agg(['mean', 'count']), '\n')

# Peek at notes, and check for leakage keywords
for n in df['patient_notes'].sample(3, random_state=1):
    print(n, '\n')
print(df['patient_notes'].str.contains(
    r'discharg|length of stay|hospital day|expected stay', case=False).mean())

count    10000.00000
mean         7.98500
std          4.01188
min          1.00000
25%          5.00000
50%          7.00000
75%         10.00000
max         33.00000
Name: los, dtype: float64
0 rows with LOS <= 0
    date_of_admission date_of_discharge
min        2024-05-15        2024-05-18
max        2025-05-14        2025-06-10
los
1      121
2      201
3      466
4      726
5     1031
6     1222
7     1298
8     1272
9     1040
10     771
11     559
12     405
13     262
14     132
15      91
16      56
17      34
18      30
19      40
20      33
21      32
22      30
23      25
24      34
25      15
26      21
27      19
28      19
29       4
30       5
31       4
32       1
33       1
Name: count, dtype: int64
gender
Female    5062
Male      4726
Other      212
Name: count, dtype: int64 

race_ethnicity
Caucasian           6025
African American    1543
Hispanic/Latino     1175
Asian                769
Native American      283
Other                205
Name: count, dtype: int64 




---



## Part 2: Using unstructured patient notes with transformer language models for NLP.



---



## Part 3: Reflective questions about ethical considerations and concerns.